In [1]:
import os
import pandas as pd
import boto3
from tqdm import tqdm

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.0' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


### Functions

In [2]:
def copy_s3_file(str_bucket, str_key_source, str_key_destination):
    cls_client = boto3.client('s3')
    dict_source = {
        'Bucket': str_bucket,
        'Key': str_key_source,
    }
    cls_client.copy_object(
        Bucket=str_bucket, # bucket
        Key=str_key_destination, # destination
        CopySource=dict_source, # source
    )

### Constants

In [3]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_dirname_output = './output'
str_variant = 'noPTImodel7'

Project: 20231010-gen-xii


### Output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Read best models output

In [5]:
str_filename = 'df_best_models.csv'
str_uri = f's3://{str_project}/09_early_indicators/{str_variant}/best_models/{str_filename}'
df = pd.read_csv(str_uri)
df

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:272: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


,target,iteration,learning_rate,flt_eval_metric_train,flt_eval_metric_valid,diff,best_iteration
0,DQ30_0,87,0.8780,0.991305,0.999951,0.008646,19
1,DQ30_1,90,0.9083,0.830589,0.948104,0.117515,18
2,Default_2,66,0.6663,0.779115,0.941510,0.162394,6
3,DQ15_0,88,0.8881,0.892957,0.808604,0.084353,12
4,DQ1_24,4,0.0413,0.817134,0.761187,0.055947,647
...,...,...,...,...,...,...,...
136,Default_4,29,0.2933,0.813136,0.656161,0.156975,6
137,Default_3,6,0.0615,0.916739,0.654901,0.261839,30
138,DQ1_2,2,0.0212,0.717251,0.651871,0.065380,999
139,DQ1_1,4,0.0413,0.726654,0.628821,0.097833,601


### Create dictionary

In [6]:
dict_best_models = dict(zip(df['target'], df['iteration']))
dict_best_models

{'DQ30_0': 87,
 'DQ30_1': 90,
 'Default_2': 66,
 'DQ15_0': 88,
 'DQ1_24': 4,
 'DQ1_23': 6,
 'DQ1_22': 3,
 'DQ1_21': 5,
 'DQ60_21': 4,
 'DQ60_19': 4,
 'DQ60_22': 4,
 'DQ1_20': 2,
 'DQ1_19': 2,
 'DQ60_20': 4,
 'DQ60_23': 5,
 'DQ1_17': 5,
 'DQ90_21': 5,
 'DQ60_24': 3,
 'DQ1_18': 4,
 'DQ90_23': 3,
 'DQ90_22': 8,
 'DQ60_18': 4,
 'DQ90_20': 5,
 'DQ60_17': 5,
 'DQ90_24': 3,
 'DQ1_16': 5,
 'DQ30_20': 4,
 'DQ30_21': 4,
 'DQ30_24': 3,
 'DQ30_19': 4,
 'DQ30_23': 8,
 'DQ1_15': 7,
 'DQ30_22': 5,
 'DQ90_17': 3,
 'DQ30_18': 4,
 'DQ90_16': 8,
 'DQ60_16': 4,
 'DQ90_15': 2,
 'DQ90_19': 4,
 'DQ60_15': 4,
 'DQ1_14': 3,
 'DQ1_13': 6,
 'DQ30_17': 3,
 'Default_24': 5,
 'DQ30_14': 3,
 'DQ90_18': 5,
 'DQ30_15': 4,
 'DQ60_14': 2,
 'DQ15_24': 4,
 'DQ30_16': 3,
 'DQ60_13': 4,
 'DQ30_13': 6,
 'DQ90_10': 7,
 'DQ90_14': 8,
 'DQ15_23': 5,
 'DQ15_22': 8,
 'Default_21': 6,
 'DQ15_21': 4,
 'Default_22': 4,
 'Default_23': 3,
 'DQ15_20': 3,
 'Default_20': 7,
 'DQ30_12': 3,
 'Default_19': 3,
 'DQ15_19': 5,
 'DQ90_9': 8,
 '

### Iterate and save

In [7]:
for str_target, int_iteration in tqdm(dict_best_models.items()):
    # get source info
    str_filename = f'dict_model_inference_{int_iteration}.pkl'
    str_key_source = f'09_early_indicators/{str_variant}/models/{str_target}/{str_filename}'
    # get destination info
    str_filename = 'dict_model_inference.pkl'
    str_key_destination = f'09_early_indicators/{str_variant}/best_models/models/{str_target}/{str_filename}'
    # copy file
    copy_s3_file(
        str_bucket=str_project, 
        str_key_source=str_key_source, 
        str_key_destination=str_key_destination,
    )

100%|██████████| 141/141 [00:36<00:00,  3.84it/s]
